# Week 82, Day 2: Observability

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w082d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
server_a = [10, 12, 11, 13, 900]          # p99 here is 900
server_b = [10, 10, 11, 12, 12] * 20      # p99 here is 12
average_of_p99s = (900 + 12) / 2
combined = sorted(server_a + server_b)
import math
true_p99 = combined[math.ceil(0.99 * len(combined)) - 1]
print(average_of_p99s, true_p99)

In [ ]:
buckets = [(0.1, 240), (0.5, 290), (1.0, 297), (float("inf"), 300)]   # cumulative counts

def quantile(q, buckets):
    rank = q * buckets[-1][1]
    lower_bound, lower_count = 0.0, 0
    for bound, count in buckets:
        if count >= rank:
            if bound == float("inf"):
                return lower_bound
            return lower_bound + (bound - lower_bound) * (rank - lower_count) / (count - lower_count)
        lower_bound, lower_count = bound, count

for q in (0.5, 0.9, 0.99):
    print(f"p{int(q * 100)} is about {quantile(q, buckets):.3f} s")

> **Key idea.** Logs record events (details of one request), metrics aggregate numbers (dashboards and alerts, cheap if labels are bounded), traces follow one request through every service (where the time went; usually sampled); link them with request and trace ids. Counters only rise (use rates, treating drops as resets), gauges go up and down, histograms count observations into cumulative le buckets with sum and count; histograms add across servers, and percentiles are estimated by linear interpolation in the bucket holding the target rank. Never average percentiles. Each label-value combination is a series: keep labels bounded (route templates, status classes; never user ids or raw URLs). Spans have trace id, span id, parent, start and end; self time is duration minus children; the critical path is what to speed up. Propagate context with traceparent (00-trace-parent-flags); head sampling decides at the start, tail sampling keeps slow and failed traces. Watch RED for services and USE for resources; alert on symptoms.

### Exercise: Counters, gauges and histograms

Write a class `Registry` with `counter(name, value=1, **labels)` (add to a counter; a negative value raises `ValueError("counters only go up")`), `gauge(name, value, **labels)` (set a gauge) and `observe(name, value, buckets, **labels)` (record one observation in a histogram with those upper bounds, a sorted list of floats). Then `expose()`: the text format, one metric name at a time in name order, each starting with `# TYPE NAME counter|gauge|histogram`, then its series sorted by their label text, where labels are written `{a="1",b="2"}` (sorted by label name; nothing at all when there are no labels). Counter and gauge lines are `NAME{LABELS} VALUE`. Histograms give, for each bucket bound and then `+Inf`, `NAME_bucket{LABELS,le="BOUND"} CUMULATIVE_COUNT` (the `le` label goes last; bounds written with `repr`-style shortest form via `str(float)`), then `NAME_sum{LABELS} SUM` and `NAME_count{LABELS} COUNT`. Values are written with `str()`; join the lines with newlines.

In [ ]:
class Registry:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_expose():\n    \"the text format\"\n    r = Registry()\n    r.counter(\"http_requests_total\", method=\"GET\", status=\"200\")\n    r.counter(\"http_requests_total\", 2, method=\"GET\", status=\"200\")\n    r.counter(\"http_requests_total\", method=\"POST\", status=\"500\")\n    r.gauge(\"queue_length\", 7)\n    r.gauge(\"queue_length\", 3)\n    for v in (0.05, 0.2, 0.7, 3.0):\n        r.observe(\"request_seconds\", v, [0.1, 0.5, 1.0], route=\"/tickets\")\n    assert r.expose().split(\"\\n\") == [\n        \"# TYPE http_requests_total counter\",\n        'http_requests_total{method=\"GET\",status=\"200\"} 3',\n        'http_requests_total{method=\"POST\",status=\"500\"} 1',\n        \"# TYPE queue_length gauge\",\n        \"queue_length 3\",\n        \"# TYPE request_seconds histogram\",\n        'request_seconds_bucket{route=\"/tickets\",le=\"0.1\"} 1',\n        'request_seconds_bucket{route=\"/tickets\",le=\"0.5\"} 2',\n        'request_seconds_bucket{route=\"/tickets\",le=\"1.0\"} 3',\n        'request_seconds_bucket{route=\"/tickets\",le=\"+Inf\"} 4',\n        'request_seconds_sum{route=\"/tickets\"} 3.95',\n        'request_seconds_count{route=\"/tickets\"} 4']\n    try:\n        r.counter(\"http_requests_total\", -1)\n    except ValueError as e:\n        assert str(e) == \"counters only go up\"\n    else:\n        raise AssertionError(\"negative\")\n    bare = Registry()\n    bare.observe(\"x\", 1, [1.0])\n    assert bare.expose().split(\"\\n\")[1:3] == ['x_bucket{le=\"1.0\"} 1', 'x_bucket{le=\"+Inf\"} 1']")

### Exercise: Percentiles from buckets

Write `merge(*histograms)`: each histogram is a list of `(upper_bound, cumulative_count)` pairs with the same bounds, the last being `float("inf")`; return their bucket-by-bucket sum. Then `quantile(q, buckets)`: estimate the `q`-quantile (0 to 1) like Prometheus's `histogram_quantile`: the target rank is `q × total`; find the first bucket whose cumulative count is at least the rank, and interpolate linearly between that bucket's lower bound (the previous bucket's bound, or 0 for the first) and its upper bound, by where the rank falls between the previous cumulative count and this one. If the rank falls in the `+Inf` bucket, return the highest finite bound. With no observations, return `None`. Round to 4 decimal places.

In [ ]:
def merge(*histograms):
    ___

def quantile(q, buckets):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_quantile():\n    \"interpolated inside the bucket\"\n    inf = float(\"inf\")\n    h = [(0.1, 240), (0.5, 290), (1.0, 297), (inf, 300)]\n    assert quantile(0.5, h) == 0.0625 and quantile(0.9, h) == 0.34 and quantile(0.99, h) == 1.0\n    assert quantile(0.98, h) == 0.7857 and quantile(0.5, [(0.1, 0), (inf, 0)]) is None\n    a = [(0.1, 5), (0.5, 5), (inf, 5)]\n    b = [(0.1, 100), (0.5, 100), (inf, 100)]\n    assert merge(a, b) == [(0.1, 105), (0.5, 105), (inf, 105)]\n    slow = [(0.1, 0), (0.5, 0), (1.0, 5), (inf, 5)]\n    assert quantile(0.99, slow) == 0.995 and quantile(0.5, merge(slow, [(0.1, 95), (0.5, 95), (1.0, 95), (float(\"inf\"), 95)])) == 0.0526")

### Exercise: Rates from counters that reset

A counter is scraped as a list of `(time, value)` samples. Write `increase(samples)`: the total increase over the samples, where a value **lower** than the previous one means the process restarted (the counter reset to 0), so the increase for that step is the new value itself. Then `rate(samples)`: the increase per second over the time from the first sample to the last (`0.0` if there are fewer than two samples), rounded to 3 places. And `error_ratio(errors, total)`: the rate of one counter over the rate of the other, rounded to 4 places (`0.0` when there's no traffic).

In [ ]:
def increase(samples):
    ___

def rate(samples):
    ___

def error_ratio(errors, total):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_rates():\n    \"resets are not negative traffic\"\n    s = [(0, 100), (15, 160), (30, 220), (45, 10), (60, 70)]\n    assert increase(s) == 190 and rate(s) == 3.167\n    assert rate([(0, 5)]) == 0.0 and increase([]) == 0\n    errors = [(0, 2), (60, 5)]\n    total = [(0, 400), (60, 1000)]\n    assert error_ratio(errors, total) == 0.005 and error_ratio(errors, [(0, 10), (60, 10)]) == 0.0")

### Exercise: Where the time went

Spans are dicts with `id`, `parent` (`None` for the root), `name`, `start` and `end` (milliseconds). Write `tree(spans)`: lines showing the trace, each `INDENT NAME DURATIONms`, children indented two spaces more than their parent and ordered by start time, starting from the root. `self_times(spans)`: a dict from span name to its duration minus the total duration of its direct children. And `critical_path(spans)`: from the root, repeatedly follow the child that **ends last** (ties to the earliest start), returning the list of names.

In [ ]:
def tree(spans):
    ___

def self_times(spans):
    ___

def critical_path(spans):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("SPANS = [\n    {\"id\": \"1\", \"parent\": None, \"name\": \"GET /checkout\", \"start\": 0, \"end\": 340},\n    {\"id\": \"4\", \"parent\": \"1\", \"name\": \"payments.charge\", \"start\": 61, \"end\": 330},\n    {\"id\": \"2\", \"parent\": \"1\", \"name\": \"auth.verify\", \"start\": 2, \"end\": 14},\n    {\"id\": \"3\", \"parent\": \"1\", \"name\": \"cart.load\", \"start\": 15, \"end\": 60},\n    {\"id\": \"5\", \"parent\": \"3\", \"name\": \"db.query\", \"start\": 18, \"end\": 55},\n    {\"id\": \"6\", \"parent\": \"4\", \"name\": \"bank.api\", \"start\": 70, \"end\": 320},\n]\n\ndef test_tree():\n    \"the shape of a request\"\n    assert tree(SPANS) == [\"GET /checkout 340ms\", \"  auth.verify 12ms\", \"  cart.load 45ms\", \"    db.query 37ms\",\n                           \"  payments.charge 269ms\", \"    bank.api 250ms\"]\n\ndef test_times_and_path():\n    \"self time and the critical path\"\n    assert self_times(SPANS) == {\"GET /checkout\": 14, \"payments.charge\": 19, \"auth.verify\": 12, \"cart.load\": 8, \"db.query\": 37, \"bank.api\": 250}\n    assert critical_path(SPANS) == [\"GET /checkout\", \"payments.charge\", \"bank.api\"]")

### Exercise: RED from logs

Week 80's JSON access log lines look like `{"request_id": ..., "method": "GET", "path": "/tickets/7", "status": 200, "ms": 12, "route": "/tickets/{ticket_id}"}`. Write `red(lines, window_seconds)`: parse each line with `json.loads` (skip lines that aren't valid JSON), group by `route`, and return a dict from route (sorted) to `{"rate": REQUESTS_PER_SECOND, "errors": ERROR_SHARE, "p95": P95_MS}`: the rate over the window rounded to 2 places, the share of requests with a status of 500 or more rounded to 3 places, and the p95 duration by nearest rank.

In [ ]:
import json
import math

def red(lines, window_seconds):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_red():\n    \"rate, errors, duration\"\n    import json\n    lines = []\n    for i in range(20):\n        lines.append(json.dumps({\"method\": \"GET\", \"path\": f\"/tickets/{i}\", \"status\": 500 if i == 3 else 200, \"ms\": 10 + i, \"route\": \"/tickets/{ticket_id}\"}))\n    for ms in (5, 6, 7, 400):\n        lines.append(json.dumps({\"method\": \"GET\", \"path\": \"/healthz\", \"status\": 200, \"ms\": ms, \"route\": \"/healthz\"}))\n    lines.append(\"not json\")\n    assert red(lines, 60) == {\"/healthz\": {\"rate\": 0.07, \"errors\": 0.0, \"p95\": 400},\n                              \"/tickets/{ticket_id}\": {\"rate\": 0.33, \"errors\": 0.05, \"p95\": 28}}")

### Exercise: Keeping labels bounded

Write a class `CardinalityGuard(max_series_per_metric, forbidden=("user_id", "email", "request_id", "url"))`. `allow(name, labels)` returns `True` if a series with these labels (a dict) may be recorded: any forbidden label name refuses it (`False`), as does a **new** label combination once the metric already has `max_series_per_metric` series; combinations already seen are always allowed. Count refusals per metric in `self.dropped`. Then `normalise_route(path)`: replace each path segment made only of digits, or that looks like a UUID (32 hex digits with or without hyphens in the 8-4-4-4-12 pattern), with `{id}`, so `/tickets/7/comments` becomes `/tickets/{id}/comments`.

In [ ]:
import re

class CardinalityGuard:
    ___

def normalise_route(path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_guard():\n    \"bounded series\"\n    g = CardinalityGuard(2)\n    assert g.allow(\"requests\", {\"route\": \"/a\", \"status\": \"200\"}) and g.allow(\"requests\", {\"route\": \"/b\", \"status\": \"200\"})\n    assert not g.allow(\"requests\", {\"route\": \"/c\", \"status\": \"200\"}) and g.allow(\"requests\", {\"status\": \"200\", \"route\": \"/a\"})\n    assert not g.allow(\"logins\", {\"user_id\": \"7\"}) and g.allow(\"logins\", {\"result\": \"ok\"})\n    assert g.dropped == {\"requests\": 1, \"logins\": 1}\n\ndef test_routes():\n    \"ids become placeholders\"\n    assert normalise_route(\"/tickets/7/comments\") == \"/tickets/{id}/comments\"\n    assert normalise_route(\"/users/550e8400-e29b-41d4-a716-446655440000\") == \"/users/{id}\"\n    assert normalise_route(\"/files/550e8400e29b41d4a716446655440000/raw\") == \"/files/{id}/raw\"\n    assert normalise_route(\"/v2/tickets\") == \"/v2/tickets\" and normalise_route(\"/\") == \"/\"")

### Bug hunt: Averaged percentiles

The fleet dashboard averages each server's p99, which gives a number no request experienced: this prints `fleet p99: 456.0`. Merge the servers' raw latencies and take the p99 of all of them (nearest rank), so it prints `fleet p99: 13`.

In [ ]:
import math

servers = {"a": [10, 12, 11, 13, 900], "b": [10, 10, 11, 12, 12] * 20}

def p99(values):
    ordered = sorted(values)
    return ordered[math.ceil(0.99 * len(ordered)) - 1]

print("fleet p99:", sum(p99(v) for v in servers.values()) / len(servers))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"percentiles don't average\"\n    expect_output(\"fleet p99: 13\")")

### Bug hunt: Negative traffic

A process restarted between two scrapes, so its counter dropped from 900 to 30, and the rate calculation reports a **negative** request rate: `requests/s: -14.5`. Treat a drop as a reset (the increase is the new value), so it prints `requests/s: 0.5`.

In [ ]:
samples = [(0, 900), (60, 30)]
(t0, v0), (t1, v1) = samples
print("requests/s:", (v1 - v0) / (t1 - t0))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"resets\"\n    expect_output(\"requests/s: 0.5\")")

### Bug hunt: A label that exploded

The request counter is labelled with the raw path, so every ticket id creates a new series: `1000 series`. Label it with the route template instead (replace digit-only segments with `{id}`), so it prints `1 series`.

In [ ]:
from collections import Counter

series = Counter()
for ticket_id in range(1000):
    path = f"/tickets/{ticket_id}"
    series[("GET", path)] += 1
print(len(series), "series")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"bounded labels\"\n    expect_output(\"1 series\")")

### Puzzle: Recording a request

Arrange the observability work around one request: start a span, time it, and record RED metrics with bounded labels. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # "counter", "gauge" or "histogram": requests in progress right now
q = ___   # the RED letter that stands for latency
r = ___   # the series for labels route (20 values) times status (5 values)
s = ___   # the W3C header that carries trace context
t = ___   # "head" or "tail" sampling: keeps every slow or failed trace

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"gauge\" and q in (\"D\", \"duration\", \"Duration\") and r == 100, \"p, q and r\"\n    assert s == \"traceparent\" and t == \"tail\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a tracer with propagation

Write a class `Tracer(service, sample_rate, ids)`: `ids` is a function returning random hex strings (`ids(32)` for trace ids, `ids(16)` for span ids), so tests can control them. `start_span(name, now, traceparent=None)`: if `traceparent` is given and valid (`00-` then 32 hex, `-`, 16 hex, `-`, 2 hex), continue that trace: the new span's parent is the header's span id and it's sampled if the flags are `01`; an invalid or missing header starts a new trace with a new trace id, no parent, and a **head sampling** decision: sampled if `int(trace_id[:8], 16) / 0xFFFFFFFF < sample_rate`. Return a span dict `{"trace_id", "span_id", "parent_id", "name", "service", "start", "end": None, "sampled", "error": False}`. `inject(span)` returns the `traceparent` header for calls made from that span. `finish(span, now, error=False)` sets `end` and `error`, and **exports** the span (adds it to `self.exported`) if it's sampled. Then `tail_keep(spans, slow_ms)`: from a list of finished spans (any traces), return the sorted ids of traces that have any span with an error or any span lasting more than `slow_ms`.

In [ ]:
import re

class Tracer:
    ___

def tail_keep(spans, slow_ms):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _ids(values):\n    it = iter(values)\n    return lambda n: next(it)\n\nTRACE = \"0af7651916cd43dd8448eb211c80319c\"\n\ndef test_propagation():\n    \"a trace across two services\"\n    web = Tracer(\"web\", 1.0, _ids([TRACE, \"b7ad6b7169203331\", \"00f067aa0ba902b7\"]))\n    root = web.start_span(\"GET /checkout\", 0)\n    assert root[\"trace_id\"] == TRACE and root[\"parent_id\"] is None and root[\"sampled\"] is True\n    header = web.inject(root)\n    assert header == f\"00-{TRACE}-b7ad6b7169203331-01\"\n    payments = Tracer(\"payments\", 0.0, _ids([\"53995c3f42cd8ad8\"]))\n    child = payments.start_span(\"charge\", 61, traceparent=header)\n    assert child[\"trace_id\"] == TRACE and child[\"parent_id\"] == \"b7ad6b7169203331\" and child[\"sampled\"] is True and child[\"service\"] == \"payments\"\n    payments.finish(child, 330)\n    web.finish(root, 340, error=False)\n    assert child[\"end\"] == 330 and payments.exported == [child] and web.exported == [root]\n\ndef test_sampling():\n    \"head sampling, unsampled headers and bad headers\"\n    low = \"1fffffff\" + \"0\" * 24\n    high = \"f0000000\" + \"0\" * 24\n    t = Tracer(\"web\", 0.25, _ids([low, \"a\" * 16, high, \"b\" * 16, \"c\" * 32, \"d\" * 16]))\n    s1, s2 = t.start_span(\"a\", 0), t.start_span(\"b\", 0)\n    assert s1[\"sampled\"] is True and s2[\"sampled\"] is False and t.inject(s2).endswith(\"-00\")\n    t.finish(s1, 5)\n    t.finish(s2, 5)\n    assert t.exported == [s1]\n    s3 = t.start_span(\"c\", 0, traceparent=\"00-not-valid-01\")\n    assert s3[\"trace_id\"] == \"c\" * 32 and s3[\"parent_id\"] is None\n    off = Tracer(\"x\", 1.0, _ids([\"e\" * 16]))\n    child = off.start_span(\"x\", 0, traceparent=f\"00-{TRACE}-{'f' * 16}-00\")\n    off.finish(child, 1)\n    assert child[\"sampled\"] is False and off.exported == []\n\ndef test_tail():\n    \"keep the interesting traces\"\n    spans = [{\"trace_id\": \"t1\", \"start\": 0, \"end\": 50, \"error\": False}, {\"trace_id\": \"t2\", \"start\": 0, \"end\": 900, \"error\": False},\n             {\"trace_id\": \"t3\", \"start\": 0, \"end\": 10, \"error\": True}, {\"trace_id\": \"t1\", \"start\": 10, \"end\": 20, \"error\": False}]\n    assert tail_keep(spans, 500) == [\"t2\", \"t3\"]")

## Warm-up

Averaging two servers' p99s gives 456 ms, a number that describes nothing: the real p99 of all requests is 13 ms, because server A handled only five of the 105. **Percentiles can't be averaged.** Metrics systems solve this by storing **histograms** (counts per latency range), which can be added across servers, and estimating percentiles from the totals.

## Three kinds of signal

| Signal | What it is | Good for | Cost |
|---|---|---|---|
| **Logs** | timestamped records of events (Week 80's JSON lines) | the details of one request or error | grows with traffic; expensive to keep and search |
| **Metrics** | numbers aggregated over time: counters, gauges, histograms | dashboards, trends, alerts | small and fixed, as long as labels are bounded |
| **Traces** | the path of one request through every service, as timed **spans** | finding which service or call is slow | usually **sampled**: a fraction of requests |

They answer different questions. "Is the error rate up?" is a metric. "Why did request `7b33fd1b` fail?" is a log. "Which of the six services this checkout touches made it slow?" is a trace. And they work best linked: Week 80's request id in every log line, the trace id in logs and traces, so you can go from a spike on a dashboard to the traces behind it to the log lines of one bad request.

## Metrics

Three kinds of metric cover nearly everything (the names are Prometheus's, now the common vocabulary):

- **Counter**: a number that only goes **up** (requests served, errors, bytes sent). The interesting value is its **rate**: the increase per second over a window. Counters reset to 0 when a process restarts, so rate calculations treat a drop as a reset and count the new value as the increase.
- **Gauge**: a number that goes up and down (requests in progress, queue length, memory used). Its current value is what matters.
- **Histogram**: counts of observations falling at or below each of several **bucket** bounds (`le` means "less than or equal": `le="0.1"` counts requests that took at most 0.1 seconds), plus their `sum` and `count`. Buckets are **cumulative**: each includes everything below it, and the last bucket, `le="+Inf"`, is the total. Histograms from different servers add up bucket by bucket, and percentiles are **estimated** from them by finding the bucket the target rank falls in and interpolating linearly inside it.

Each metric can have **labels** (`method="GET"`, `route="/tickets"`, `status="200"`), and each distinct combination of label values is a separate **time series** stored separately. Monitoring systems typically **scrape** each process's metrics every 15 seconds or so, in a simple text format:

```
# TYPE http_requests_total counter
http_requests_total{method="GET",route="/tickets",status="200"} 1027
# TYPE request_seconds histogram
request_seconds_bucket{le="0.1"} 240
request_seconds_bucket{le="0.5"} 290
request_seconds_bucket{le="+Inf"} 300
request_seconds_sum 41.7
request_seconds_count 300
```

**Cardinality.** The number of series is the product of each label's distinct values. `route` (20 values) times `status` (10) is 200 series: fine. Add `user_id` (a million values) and it's 200 million, enough to bring a monitoring system down. **Labels must have a small, bounded set of values**: route templates (`/tickets/{id}`, not `/tickets/7`), status classes, never user ids, emails, raw URLs or request ids. Those belong in logs and traces.

## Traces

A **trace** follows one request through every service it touches. It's made of **spans**: each span is one unit of work (an HTTP request handled, a database query, a call to another service) with a name, a start and end time, a **trace id** (the same for every span in the request) and a **parent span id** (the span that caused it). Assembled into a tree, the spans show exactly where the time went:

```
GET /checkout                    0 ms ──────────────────────────── 340 ms
  auth.verify                    2 ms ──  14 ms
  cart.load                     15 ms ────── 60 ms
    db.query                    18 ms ──── 55 ms
  payments.charge               61 ms ────────────────── 330 ms
```

Two numbers matter for each span: its **duration**, and its **self time** (its duration minus the time covered by its children), which is the time spent in that span's own code. The **critical path** is the chain of spans that determined the total time: here, the request waited on `payments.charge`, so making `auth.verify` faster would change nothing.

**Propagation.** For spans in different services to join one trace, the trace id and the caller's span id must travel with each request. The W3C standard header is **`traceparent`**: `00-{32 hex trace id}-{16 hex parent span id}-{2 hex flags}`, where flag `01` means "sampled". Each service reads it, starts its own span as a child, and sends a new `traceparent` (with its own span id) on its outgoing calls.

**Sampling.** Recording every span of every request is expensive, so traces are **sampled**. **Head sampling** decides at the start (keep 10% of traces, chosen from the trace id so every service makes the same decision); **tail sampling** decides at the end, keeping every trace that was slow or had an error, which are the ones people want.

## What to watch

Two simple checklists say which metrics every component should have:

- **RED**, for every service (request-driven): **R**ate (requests per second), **E**rrors (the share that fail), **D**uration (the latency distribution: p50, p95, p99).
- **USE**, for every resource (CPU, memory, disk, a connection pool): **U**tilisation (how busy), **S**aturation (how much work is waiting: a queue length), **E**rrors.

Alert on **symptoms** users feel (error rate, latency, from RED), not on causes (high CPU is fine if requests are fast). Tomorrow turns this into **SLOs** and error budgets.

### On your own computer

**OpenTelemetry** is the standard library for all three signals (`pip install opentelemetry-sdk`), with automatic instrumentation for Flask, FastAPI, requests and database drivers. **Prometheus** scrapes and stores metrics (and the `prometheus-client` package exposes them from Python with the exact counter, gauge and histogram types above); **Grafana** draws dashboards; **Jaeger** or **Tempo** store and show traces; Python's `logging` with a JSON formatter writes the logs.

## Practice

## Quiz

## Daily challenge

## Before you go

Open the network tab of your browser's developer tools on a large site and look for `traceparent` or similar trace headers on its requests; many sites send them. Tomorrow: **reliability**: SLOs and error budgets, redundancy, and the patterns (timeouts, retries, circuit breakers) that keep one failure from becoming many.